In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

from torch.utils.data import DataLoader
import torchvision.transforms as transforms


In [4]:
transforms = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

In [ ]:
trainset = CIFAR10(root='/home/prakhar/PrimeDeepLearning/CNN/data', train=True, download=True, transform=transforms)
testset = CIFAR10(root='/home/prakhar/PrimeDeepLearning/CNN/data', train=False, download=True, transform=transforms) ## keep download True when you directly want to download the dataset

100%|██████████| 170M/170M [01:02<00:00, 2.74MB/s] 


In [7]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [10]:
## define the model

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 1),


            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 1),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # for flattening
        x = self.fc_layers(x)
        return x


In [11]:
model = CNN()
optim = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 100
for e in range(epochs):
    epoch_training_loss = 0.0

    for img, lab in trainloader:
        optim.zero_grad()
        output = model.forward(img)
        loss = criterion(output, lab)
        loss.backward()
        optim.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {e+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

    

In [ ]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for img, lab in testloader:
        out = model.forward(img)
        _, predicted = torch.max(out, 1)

        correct_labels += (predicted == lab).sum().item()
        total_labels += lab.size(0)
print(f"Accuracy = {correct_labels/total_labels}")